In [ ]:
import numpy as np
from deepfmkit.physics import IfoConfig, LaserConfig, SimConfig, SignalGenerator
from deepfmkit.plotting import default_rc
import matplotlib.pyplot as plt

plt.rcParams.update(default_rc)

In [ ]:
ifo_config = IfoConfig(label="Test_Cavity")
ifo_config.ref_arml = 0.1
ifo_config.meas_arml = 0.15
ifo_config.arml_mod_amp = 1e-6
ifo_config.arml_mod_f = 5

laser_config = LaserConfig(label="Test_Laser")
laser_config.fm = 1000.0
laser_config.set_df_for_effect(ifo_config, 6.54321)

laser_config.f_n = 1e5
laser_config.amp_n = 1e-5
ifo_config.arml_n = 1e-9

In [ ]:
f_samp = int(200) * laser_config.fm

main_channel = SimConfig(
    label="Main_Measurement",
    laser_config=laser_config,
    ifo_config=ifo_config,
    f_samp=f_samp,
)

In [ ]:
sg = SignalGenerator()

# We want to generate 50 cycles of the 1 kHz modulation signal
num_cycles_to_generate = 50
duration_s = num_cycles_to_generate / main_channel.laser.fm

# Run the generator
sim_output = sg._generate_with_asd(
    main_config=main_channel,
    n_seconds=duration_s,
    trial_num=42,
)

# Extract the result object
raw_output = sim_output["main"]

In [ ]:
final_time_axis_ms = (np.arange(main_channel.N) / main_channel.f_samp) * 1000

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=False)
plt.subplots_adjust(hspace=0.4)

ax = axes[0]
ax.plot(
    final_time_axis_ms,
    raw_output.a_noise * 1e3,
    label="Amplitude noise (mV)",
    alpha=0.8,
)
ax.plot(
    final_time_axis_ms,
    raw_output.l_noise * 1e9,
    label="Armlength noise (nm)",
    alpha=0.8,
)
ax.plot(
    final_time_axis_ms, raw_output.f_noise / 1e3, label="Frequency noise (kHz)", alpha=0.8
)
ax.set_title("Input noise sources (sample)")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Noise amplitude")
ax.legend(loc="upper right")

ax = axes[1]
ax.plot(
    final_time_axis_ms, raw_output.data["ch0"], label="Simulated voltage signal $v(t)$"
)
ax.set_title("Generated DFMI signal")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Voltage (a.u.)")
ax.legend()

ax = axes[2]
ax.plot(
    final_time_axis_ms, raw_output.phi_sim, label="Ground truth phase $\\Phi_{GT}(t)$"
)
ax.plot(
    final_time_axis_ms, raw_output.phi, label="Total noisy phase $\\Phi_{total}(t)$"
)
ax.set_title("Underlying phase components")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Phase (rad)")
ax.legend()

plt.tight_layout()
plt.show()